# Next step: step 8b, saving and loading
**What it adds:** `run.save`. `simr.run(models, request, output=folder)` writes the requested data to one folder per
model, and `simr.load_model` / `simr.load_simulation` read it back exactly as `run` returns it. With an output folder,
data that's already saved is loaded instead of computed.

```
<output>/model_000000/
    model.json          the model's record: 14 named parameters, and each saved simulation by name
                        ("sim_00" for simulator entry 0): config (seed included), timing, saved names, file, code
    simulations/
        sim_00_particles.npy   one file per saved array, <simulation>_<item>.npy, float32, no compression
        sim_00_orbit.npy
    images/             later
```

| | What it is |
|---|---|
| `"run": {"save": [...]}` | Names to write, from the same list as `return`. Independent of `return`: a run can do both |
| `simr.run(..., output=folder)` | Where to save, and to load saved data from. A runtime argument, not part of the request: it doesn't change any result |
| `simr.load_model(folder)` | The model's `model.json` as a dict |
| `simr.load_simulation(folder, "sim_00")` | That simulation's arrays, by name |
| `sim.jspam_version()`, `simr.code_versions()` | The code a simulation was made with, recorded in `model.json` |

**Rules:**
- Each `.npy` is written to a temporary name and renamed into place; only then is `model.json` (written the same way)
  updated to list it. An array counts as saved only if `model.json` lists it.
- **Reuse:** a simulation `model.json` lists under the same name (`sim_00` for entry 0) with exactly the same config
  counts as already saved. Anything requested that it has is loaded instead of computed, nothing it has is written
  again, and arrays it's missing (e.g. `sim.unperturbed` asked for later) are computed and added. A different config is
  neither used nor changed (a warning is logged).
- Code versions are recorded in `model.json` but not compared; higher-level code can check them.
- A model folder that holds a different model is an error, so two models' simulations can't mix.
- Saved but not returned means not kept in memory.
- `sim.timing` always goes in `model.json` (it's small); it has no `.npy`.

## What changed
**`src/galactic_tidal_constraints/simr/simr.py`.** The loop, now with saving and reuse. What's saved is loaded, the
rest is computed, and only what isn't saved yet is written:
```python
for m, params in enumerate(models):
    folder = Path(output) / f"model_{m:06d}" if output is not None else None
    record = _open_record(folder, params) if folder is not None else None   # model.json, or a new record
    for s, config in enumerate(configs):
        name = f"sim_{s:02d}"
        stored, new = _saved_data(folder, record, name, config) if record is not None else ({}, True)
        usable = stored or {}   # None: saved with a different config, so nothing is used or added
        values = {item: usable[item]() for item in returned if item in usable}   # load what's saved
        to_save = [] if stored is None else [item for item in saved if item not in usable]
        if new and saved:
            to_save.append("sim.timing")
        for item in dict.fromkeys(returned + to_save):
            if item not in values:
                values[item] = SIM_DATA[item](params, config)   # compute the rest
        for item in returned:
            data[item][m, s] = values[item]
        if to_save:
            _save_simulation(folder, record, name, config, ..., values.get("sim.timing"), versions)
```
`_saved_data` returns a loader for each array the simulation has (and its timing), but only if its saved config is
exactly this one. Saving: each array first, then the record:
```python
def _save_simulation(folder, record, name, config, arrays, timing, versions):
    entry = record["simulations"].get(name) or {"config": ..., "timing": ..., "files": {}, "code": versions}
    for item, array in arrays.items():
        file = array_file(name, item)                        # "simulations/sim_00_particles.npy"
        _write_atomically(folder / file, lambda f, array=array: np.save(f, array))
        entry["files"][item] = file
    record["simulations"][name] = entry
    _write_atomically(folder / "model.json", lambda f: f.write(text))

def _write_atomically(path, write):
    temporary = path.with_name(path.name + ".tmp")
    with open(temporary, "wb") as f:
        write(f)
        f.flush()
        os.fsync(f.fileno())        # on disk before the rename
    os.replace(temporary, path)     # one step: path is complete or absent, never half-written
```
**`simulation.py`:** `jspam_version()` reads `jspam`'s install record (`direct_url.json`): the repository and commit.
`simr.code_versions()` adds this repo's commit, and whether `src/`, `pyproject.toml` or `uv.lock` have uncommitted
changes (notebooks and docs don't change results).

**Tests:** `tests/test_simr.py`, section "Step 8b". **Decisions:** `PLAN.md`, Structure (one folder per model, write order).

In [ ]:
%matplotlib inline
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import simr
from galactic_tidal_constraints.simr import simulation as sim

# Line 3 of JSPAM's archive/data.txt, and the same with mass2 doubled
params = np.array([1.08119, -1.66798, -2.06826, -0.18438, -0.39693, -0.02919, 0.51654, 0.36192, 0.94043, 0.43877,
                   12.97415, 135.11902, 97.54035, 305.01225])
heavier = params.copy()
heavier[sim.PARAM_INDEX["mass2"]] *= 2
models = np.stack([params, heavier])
N = 1500

output = Path(tempfile.mkdtemp(prefix="simr_8b_"))  # a fresh folder inside the container
print("output:", output)


def show_tree(folder):
    for path in sorted(folder.rglob("*")):
        size = f"{path.stat().st_size / 1e3:8.1f} kB" if path.is_file() else ""
        print(f"{str(path.relative_to(folder)):45s} {size}")

## 1. Save and return in one run
Particles and orbits are saved for both entries; only the orbit is returned:

In [ ]:
request = {
    "run": {"return": ["sim.orbit"], "save": ["sim.particles", "sim.orbit"]},
    "simulator": [{"n1": N, "n2": N, "lnl": 0.0025}, {"n1": N, "n2": N, "lnl": 0.001}],
}
result = simr.run(models, request, output=output)
print("returned:", list(result.data))
show_tree(output)

## 2. The model's record

In [ ]:
record = simr.load_model(output / "model_000000")
print("model:", record["model"])
for name, entry in record["simulations"].items():
    print(f"\n{name}: files {entry['files']}")
    print("  config:", {key: entry["config"][key] for key in ("seed", "n1", "n2", "lnl", "start")}, "...")
    print("  timing:", {key: entry["timing"][key] for key in ("t_start", "n_steps", "start", "closest_distance")})
    print("  code:  ", entry["code"])

## 3. Loading gives back what `run` returned
`load_simulation` returns the saved arrays by name, float32. The orbit was both returned and saved, so the two can be
compared directly:

In [ ]:
for m in range(len(models)):
    for s in range(len(result.simulators)):
        loaded = simr.load_simulation(output / f"model_{m:06d}", f"sim_{s:02d}")
        same = np.array_equal(loaded["sim.orbit"], result.data["sim.orbit"][m, s])
        print(f"[{m}, {s}]  {sorted(loaded)}  particles {loaded['sim.particles'].shape} {loaded['sim.particles'].dtype}   orbit same as returned: {same}")

Plotted from disk, with no simulation run in this cell:

In [ ]:
fig, axes = plt.subplots(len(models), len(result.simulators), figsize=(12, 12), sharex=True, sharey=True, squeeze=False)
for m in range(len(models)):
    record = simr.load_model(output / f"model_{m:06d}")
    for s, (name, entry) in enumerate(record["simulations"].items()):
        loaded = simr.load_simulation(output / f"model_{m:06d}", name)
        p, orbit, n1 = loaded["sim.particles"], loaded["sim.orbit"], entry["config"]["n1"]
        ax = axes[m, s]
        ax.scatter(p[:n1, 0], p[:n1, 1], s=1)
        ax.scatter(p[n1:-1, 0], p[n1:-1, 1], s=1)
        ax.plot(orbit[:, 0], orbit[:, 1], color="k", lw=1)
        ax.set_title(f"model_{m:06d}/{name}: mass2 {record['model']['mass2']:.3g}, lnl {entry['config']['lnl']:g}")
        ax.set_xlim(-2.5, 2.5)
        ax.set_ylim(-2.5, 2.5)
        ax.set_aspect("equal")
plt.show()

## 4. Adding to saved simulations
The same request again, now also saving `sim.unperturbed`. `sim_00` and `sim_01` gain just that array; their particles
and orbits aren't rewritten:

In [ ]:
files_before = {path.name: path.stat().st_mtime_ns for path in (output / "model_000000" / "simulations").iterdir()}
simr.run(models, {**request, "run": {"save": ["sim.particles", "sim.orbit", "sim.unperturbed"]}}, output=output)
show_tree(output / "model_000000")
files_after = {path.name: path.stat().st_mtime_ns for path in (output / "model_000000" / "simulations").iterdir()}
print("rewritten:", [name for name in files_before if files_after[name] != files_before[name]])

A different entry 0 (`lnl` 0.1) and a new entry 2: `sim_00` was saved with another config, so it's left alone; only
`sim_02` is added:

In [ ]:
before = simr.load_model(output / "model_000000")
again = {
    "run": {"save": ["sim.particles", "sim.orbit"]},
    "simulator": [{"n1": N, "n2": N, "lnl": 0.1}, {"n1": N, "n2": N, "lnl": 0.001}, {"n1": N, "n2": N, "seed": 1}],
}
simr.run(models, again, output=output)
after = simr.load_model(output / "model_000000")
print("sim_00 unchanged:", after["simulations"]["sim_00"] == before["simulations"]["sim_00"],
      "  its lnl:", after["simulations"]["sim_00"]["config"]["lnl"])
print("simulations now:", list(after["simulations"]),
      "  sim_02 seed:", after["simulations"]["sim_02"]["config"]["seed"])

## 5. Loading instead of computing
With the output folder, asking only to **return** particles reads them from disk. Without it, they're simulated again:

In [ ]:
import time

ask = {**request, "run": {"return": ["sim.particles"]}}
for label, folder in [("output folder: loaded", output), ("no output folder: simulated", None)]:
    start = time.perf_counter()
    got = simr.run(models, ask, output=folder)
    print(f"{label:30s} {time.perf_counter() - start:6.2f} s")
print("same particles either way:", np.array_equal(got.data["sim.particles"][1, 1],
      simr.load_simulation(output / "model_000001", "sim_01")["sim.particles"]))

## 6. What's protected

In [ ]:
attempts = {
    "save with no output folder": lambda: simr.run(models, {"run": {"save": ["sim.orbit"]}}),
    "another model in model_000000's folder": lambda: simr.run(models[::-1], {"run": {"save": ["sim.orbit"]}}, output=output),
    "load a name model.json doesn't list": lambda: simr.load_simulation(output / "model_000000", "sim_07"),
    "an unknown name in run.save": lambda: simr.run(models, {"run": {"save": ["sim.orbits"]}}, output=output),
}
for label, call in attempts.items():
    try:
        call()
    except Exception as error:
        print(f"{label:40s} -> {type(error).__name__}: {str(error).splitlines()[0]}")
print("temporary files left:", list(output.rglob("*.tmp")))